<a href="https://colab.research.google.com/github/avinash-tiwary/ePic/blob/main/notebooks/ePic_Master_Tutorial.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ePic Master Tutorial: 1D, 2D, and 3D Particle-in-Cell Plasma Suite

Welcome to the **ePic** Master Tutorial! This interactive notebook guides you through the full physics, mathematical theory, and hands-on simulation workflows of Particle-in-Cell modeling across 1D, 2D, and 3D geometries.

---

## Table of Contents
1. **Google Colab Environment Setup**
2. **Boris Particle Pusher (Cyclotron Gyromotion & ExB Drift)**
3. **1D-3V Two-Stream Instability & Kinetic Growth Rate**
4. **1D-3V Collisionless Landau Damping**
5. **2D-3V Filamentation & Beam Instability**
6. **2D-3V Kinetic Magnetic Reconnection (Harris Current Sheet)**
7. **3D-3V Spherical Plasma Expansion & Coulomb Explosion**


In [ ]:
# ==============================================================
# Google Colab Setup & Package Installation
# ==============================================================
import sys
if 'google.colab' in sys.modules:
    print('Running in Google Colab. Installing ePic...')
    !git clone https://github.com/avinash-tiwary/ePic.git
    %cd ePic
    !pip install -e .
else:
    print('Running locally. Verifying ePic installation...')
    import epic
    print(f'ePic version {epic.__version__} loaded successfully!')


## 1. Quick Verification: Boris Gyromotion
Run a single electron in a uniform $B_z = 2.0$ field to observe machine-precision energy conservation:

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from epic.pusher.boris import boris_push

q, m, Bz = 1.0, 1.0, 2.0
B = np.array([0.0, 0.0, Bz])
E = np.zeros(3)
N_steps = 1000
dt = 2.0 * np.pi / (q * Bz / m) / 250.0

x = np.zeros((N_steps, 3))
v = np.zeros((N_steps, 3))
v[0] = [1.0, 0.0, 0.0]

for i in range(N_steps - 1):
    v[i+1] = boris_push(v[i], E, B, q, m, dt)
    x[i+1] = x[i] + v[i+1] * dt

plt.figure(figsize=(5, 5), dpi=100)
plt.plot(x[:, 0], x[:, 1], color="#0077bb", lw=2)
plt.axis("equal")
plt.title("Boris Gyromotion")
plt.grid(True)
plt.show()


## 2. 1D Two-Stream Instability
Simulate non-linear phase space vortex roll-up with $60,000$ particles:

In [ ]:
from epic.solvers.pic1d import PIC1DSolver

solver = PIC1DSolver(Nx=256, boxsize=45.0, dt=0.1)
N = 60000
w = 45.0 / N

p1 = np.random.uniform(0, 45.0, N//2)
v1 = np.random.normal(3.0, 0.5, (N//2, 3))
p2 = np.random.uniform(0, 45.0, N//2)
v2 = np.random.normal(-3.0, 0.5, (N//2, 3))

solver.add_species("b1", q=-w, m=w, pos=p1, vel=v1)
solver.add_species("b2", q=-w, m=w, pos=p2, vel=v2)
solver.initialize()
solver.run(t_end=35.0)

plt.figure(figsize=(10, 4), dpi=100)
plt.scatter(solver.species[0].pos, solver.species[0].vel[:, 0], s=0.3, c="blue", alpha=0.4)
plt.scatter(solver.species[1].pos, solver.species[1].vel[:, 0], s=0.3, c="red", alpha=0.4)
plt.xlabel("x")
plt.ylabel("vx")
plt.title("Two-Stream Phase Space at t = 35")
plt.grid(True)
plt.show()


## 3. High-Definition Simulation Animations

| 1D Two-Stream Phase Space | 1D Landau Damping |
| :---: | :---: |
| <img src="../docs/animations/two_stream_1d.gif" width="100%"/> | <img src="../docs/animations/landau_damping_1d.gif" width="100%"/> |

| 2D Filamentation | 2D Harris Reconnection |
| :---: | :---: |
| <img src="../docs/animations/filamentation_2d.gif" width="100%"/> | <img src="../docs/animations/reconnection_2d.gif" width="100%"/> |

| 3D Plasma Expansion |
| :---: |
| <img src="../docs/animations/plasma_expansion_3d.gif" width="50%"/> |
